# Final (My) Implementation

In [1]:
import torch
import numpy as np

import pyvista as pv
from pyvista.core.pointset import PolyData


from torch_geometric.data import Data

from sklearn.neighbors import NearestNeighbors

In [2]:
pv.set_jupyter_backend('client')

## Read `.vtp` File

In [54]:
mesh: PolyData = pv.read('../data/drivaer_data/run_1/boundary_1.vtp')

print(type(mesh))
print(f"Points : {mesh.n_points:,}")
print(f"Cells  : {mesh.n_cells:,}")
print(f"Bounds : {mesh.bounds}")
print(f"Cell arrays  : {list(mesh.cell_data.keys())}")
print(f"Point arrays : {list(mesh.point_data.keys())}")

<class 'pyvista.core.pointset.PolyData'>
Points : 8,846,052
Cells  : 8,828,095
Bounds : BoundsTuple(x_min = -0.7651930451393127,
            x_max =  3.9384641647338867,
            y_min = -1.0102157592773438,
            y_max =  1.0102157592773438,
            z_min = -0.31759700179100037,
            z_max =  1.195139765739441)
Cell arrays  : ['CpMeanTrim', 'pMeanTrim', 'pPrime2MeanTrim', 'wallShearStressMeanTrim']
Point arrays : []


## Build the Tensors for Transolver

### `x` np.Array

In [53]:
pos     = mesh.cell_centers().points.astype(np.float32)     # (N, 3)
normals = mesh.cell_normals.astype(np.float32)                  # (N, 3)

x = np.concatenate([pos, normals], axis=1)                  # (N, 6)
print(f"`x` shape: {x.shape}")

`x` shape: (8828095, 6)


### `y` np.Array

In [44]:
mesh.cell_data

pyvista DataSetAttributes
Association     : CELL
Active Scalars  : None
Active Vectors  : None
Active Texture  : None
Active Normals  : None
Contains arrays :
    CpMeanTrim              float32    (8828095,)
    pMeanTrim               float32    (8828095,)
    pPrime2MeanTrim         float32    (8828095,)
    wallShearStressMeanTrim float32    (8828095, 3)

In [62]:
pressure = mesh["pMeanTrim"].astype(np.float32)                  # (N,)
wss      = mesh["wallShearStressMeanTrim"].astype(np.float32)    # (N, 3)

y = np.concatenate([wss, pressure[:, np.newaxis]], axis=1)  # (N, 4) = [wx, wy, wz, p]
print(f"y shape: {y.shape}")  # expect (N, 4)

y shape: (8828095, 4)


## Create PyTorch Geometric Object

In [60]:
from torch_geometric.data import Data

data = Data(
    x = torch.from_numpy(x),
    y = torch.from_numpy(y),
    pos = torch.from_numpy(pos)
)

In [61]:
data.x.mean(dim=0)

tensor([ 1.6155e+00, -1.8751e-03,  1.2563e-01, -1.3276e-02,  3.7847e-05,
         6.8500e-02])

In [6]:

# mesh.cell_data['pMeanTrim']

pyvista DataSetAttributes
Association     : CELL
Active Scalars  : None
Active Vectors  : None
Active Texture  : None
Active Normals  : None
Contains arrays :
    CpMeanTrim              float32    (8828095,)
    pMeanTrim               float32    (8828095,)
    pPrime2MeanTrim         float32    (8828095,)
    wallShearStressMeanTrim float32    (8828095, 3)

In [23]:
mesh.get_cell(0).plot(show_edges=True, line_width=3)

Dropped Escape call with ulEscapeCode : 0x03007703


Widget(value='<iframe id="pyvista-jupyter_trame__template_P_0x79205144d4c0_3" src="http://localhost:8888/trame…

In [8]:
# mesh.plot(scalars=mesh.cell_data['wallShearStressMeanTrim'][:,0], cmap='viridis')
mesh.plot(scalars=mesh.cell_data['pMeanTrim'], cmap='viridis')

Dropped Escape call with ulEscapeCode : 0x03007703


Widget(value='<iframe id="pyvista-jupyter_trame__template_P_0x79205144cda0_1" src="http://localhost:8888/trame…

## Run Mesh Pre-Processing
1. Cell -> Point
2. Triangulation
3. Decimate
4. Compute Normals

In [10]:
# Load a high-density built-in mesh
mesh = pv.examples.download_cow()
print(f"Original Triangles: {mesh.n_cells}")
mesh = mesh.triangulate()

# Decimate the mesh by 90% (keep only 10% of triangles)
decimated_mesh = mesh.decimate(target_reduction=0.9)
print(f"Decimated Triangles: {decimated_mesh.n_cells}")

# Plot both side-by-side to visually inspect the quality
plotter = pv.Plotter(shape=(1, 2))

plotter.subplot(0, 0)
plotter.add_text("Original Mesh", font_size=12)
plotter.add_mesh(mesh, show_edges=True, color='white')

plotter.subplot(0, 1)
plotter.add_text("Decimated Mesh (90% Reduction)", font_size=12)
plotter.add_mesh(decimated_mesh, show_edges=True, color='white')

plotter.link_views() # Sync camera rotation
plotter.show()

Original Triangles: 3263
Decimated Triangles: 580


Dropped Escape call with ulEscapeCode : 0x03007703


Widget(value='<iframe id="pyvista-jupyter_trame__template_P_0x79205144f3b0_2" src="http://localhost:8888/trame…

In [25]:
mesh_processed = mesh.cell_data_to_point_data().triangulate().decimate(target_reduction=0.99)

In [ ]:
print(f"Points : {mesh_processed.n_points:,}")
print(f"Cells  : {mesh_processed.n_cells:,}")

In [5]:
mesh_processed = mesh.cell_data_to_point_data().triangulate().decimate(target_reduction=0.99)

# Each decimated point is an exact original point — NN gives index into original
nbrs = NearestNeighbors(n_neighbors=1, algorithm='kd_tree').fit(mesh.points)
dists, original_indices = nbrs.kneighbors(mesh_processed.points)

print(f"Max dist (should be ~0): {dists.max():.2e}")  # sanity check
original_indices = original_indices.squeeze()  # (N_dec,)

# Now index straight into the original CFD arrays
pressure = mesh.point_data["pMeanTrim"][original_indices].astype(np.float32)
wss      = mesh.point_data["wallShearStressMeanTrim"][original_indices].astype(np.float32)

mesh_processed = mesh_processed.compute_normals(cell_normals=False, point_normals=True, auto_orient_normals=True)

Max dist (should be ~0): 5.66e-03


KeyError: 'pMeanTrim'

In [ ]:
print("Points data after processing:")
print(f"{mesh_processed.n_points:,} points, {mesh_processed.n_cells:,} cells")
for k, v in mesh_processed.point_data.items():
  print(f"  {k}: shape={v.shape}")

## Build the Tensors for Transolver

### Build the remaining attributes for PyTorch Geometric object input to Transolver
1. `edge_index: LongTensor` – Graph connectivity in COO format with shape `[2, num_edges]`
2. `pos: torch.Tensor` – Node position matrix with shape `[num_nodes, 3]`
3. `surf: torch.BooleanTensor` - Which nodes are surface nodes (in this case, all of them) with shape `[num_nodes, 1]`

In [ ]:
def get_coo_edges_from_mesh(mesh_processed: PolyData) -> np.array:
    """
    Returns an edge_index array of shape [2, num_nodes]
    First row is the indices of the source nodes
    Second row is the same for the destination nodes
    """
    faces = mesh_processed.faces.reshape(-1, 4)[:, 1:]  # (N_tri, 3); faces[0] is a list of the indices of the 3 nodes that form the triangle; pos[i] would give the actual coordinates of the point at index i
    print(f'num_faces: {face.shape[0]}')
    
    pair_idx = np.array([[0,1],[1,2],[0,2]])        # 3 edges per triangle    
    src = faces[:, pair_idx[:, 0]].reshape(-1)      # (N_tri*3,) — edge sources
    dst = faces[:, pair_idx[:, 1]].reshape(-1)      # (N_tri*3,) — edge targets
    
    
    all_src = np.concatenate([src, dst])
    all_dst = np.concatenate([dst, src])
    
    edge_index = torch.from_numpy(
      np.unique(np.stack([all_src, all_dst], axis=0), axis=1)
    ).to(torch.long)  # shape: (2, E)  — row 0 = sources, row 1 = destinations
    print(f"edge_index shape: {edge_index.shape}")

    return edge_index

### Convert to PyTorch Geometric object

In [ ]:
edge_index = get_coo_edges_from_mesh(mesh_processed)

data = Data(
  pos         =  torch.tensor(pos,          dtype=torch.float32),
  x           =  torch.tensor(x,            dtype=torch.float32),
  y           =  torch.tensor(y,            dtype=torch.float32),
  edge_index  =  edge_index,
  surf        =  torch.ones(pos.shape[0],   dtype=torch.bool),   # all surface
)

## Also build the `geom` input for the Transolver

In [ ]:
# geom = subsampled surface point cloud for the shape encoder
geom_idx  = np.random.choice(pos.shape[0], min(8192, pos.shape[0]), replace=False)
geom      = torch.tensor(pos[geom_idx], dtype=torch.float32)  # (≤8192, 3)
print(f"geom shape: {geom.shape}")